# 🔬 Validação Completa & Comparação — EfficientNet-B0 vs B4
### Projeto Integrador II | Retinopatia Diabética

---

**O que este notebook faz:**
- Detecta automaticamente quais modelos já foram salvos no Drive
- Valida cada modelo encontrado (pode rodar com só o B0, só o B4, ou os dois)
- Gera métricas completas: Accuracy, F1, AUC-ROC, Kappa
- Matrizes de confusão, Curvas ROC, F1 por classe, Radar, Análise de erros
- Salva todos os gráficos no Drive

**Basta rodar célula a célula — nenhuma configuração manual necessária.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 1 — Instalar dependências e montar Drive             ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess
subprocess.run(["pip", "install", "-q", "timm", "albumentations",
                "opencv-python", "scikit-learn", "tqdm", "seaborn"])

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Drive montado')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 2 — Imports                                          ║
# ╚══════════════════════════════════════════════════════════════╝

import os, gc, warnings, time
import torch
import torch.nn as nn
import timm
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    accuracy_score, precision_recall_fscore_support,
    cohen_kappa_score
)
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Device ────────────────────────────────────────────────────
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (device.type == 'cuda')

print(f'Dispositivo : {device}')
if device.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('✅ Imports prontos')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 3 — Configurações                                    ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Pasta raiz do dataset ─────────────────────────────────────
BASE_PATH   = '/content/drive/MyDrive/RETINOPATIA'
OUTPUT_DIR  = os.path.join(BASE_PATH, 'validacao_resultados')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Classes ───────────────────────────────────────────────────
CLASSES     = {'No_DR': 0, 'Mild': 1, 'Moderate': 2, 'Severe': 3, 'Proliferate_DR': 4}
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
N_CLASSES   = 5

# ── Configuração de cada modelo ───────────────────────────────
# O notebook detecta automaticamente quais arquivos existem.
# Ajuste apenas se seus arquivos .pth estiverem em outro local.
MODEL_CONFIGS = {
    'EfficientNet-B0': {
        # Tenta o modelo final primeiro; se não existir, usa o checkpoint
        'paths': [
            '/content/drive/MyDrive/retinopatia_model_final.pth',
            '/content/drive/MyDrive/checkpoints_retina/b0_checkpoint.pth',
        ],
        'arch'    : 'efficientnet_b0',
        'img_size': 224,
        'head'    : 'direct',   # num_classes=5 diretamente no timm
        'color'   : '#4C72B0',
    },
    'EfficientNet-B4': {
        'paths': [
            '/content/drive/MyDrive/RETINOPATIA/retinopatia_b4_final.pth',
            '/content/drive/MyDrive/RETINOPATIA/checkpoints_retina/b4_checkpoint.pth',
        ],
        'arch'    : 'efficientnet_b4',
        'img_size': 380,
        'head'    : 'custom',   # num_classes=0 + Dropout(0.4) + Linear(5)
        'color'   : '#DD8452',
    },
}

# Batch de validação — conservador para caber em qualquer GPU
VAL_BATCH   = 8
NUM_WORKERS = 2

# ── Checar quais modelos existem ──────────────────────────────
print('\n📂 Verificando arquivos de modelo:')
for name, cfg in MODEL_CONFIGS.items():
    found = None
    for p in cfg['paths']:
        if os.path.exists(p):
            found = p
            break
    if found:
        size_mb = os.path.getsize(found) / 1e6
        cfg['resolved_path'] = found
        print(f'  ✅ {name}: {found}  ({size_mb:.0f} MB)')
    else:
        cfg['resolved_path'] = None
        print(f'  ❌ {name}: não encontrado')
        print(f'     Caminhos testados: {cfg["paths"]}')

available = [n for n, c in MODEL_CONFIGS.items() if c['resolved_path']]
print(f'\n🔍 Modelos disponíveis para validação: {available}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 4 — Dataset e transformações                         ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Montar DataFrame ──────────────────────────────────────────
data = []
for c, lbl in CLASSES.items():
    folder = os.path.join(BASE_PATH, c)
    if not os.path.exists(folder):
        print(f'⚠️  Pasta não encontrada: {folder}')
        continue
    for img_file in os.listdir(folder):
        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            data.append([os.path.join(folder, img_file), lbl])

df = pd.DataFrame(data, columns=['image', 'label'])
print(f'Total de imagens encontradas: {len(df)}')
print('Distribuição por classe:')
for i, name in enumerate(CLASS_NAMES):
    n = (df['label'] == i).sum()
    print(f'  {name:<18} {n:>5} imagens')

# ── Split idêntico ao do treino (random_state=42) ─────────────
# Garante que a validação usa EXATAMENTE as mesmas imagens
# que foram excluídas do treino
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)
print(f'\nSplit (random_state=42): Treino={len(train_df)} | Validação={len(val_df)}')


# ── CLAHE ─────────────────────────────────────────────────────
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_clahe, a, b]), cv2.COLOR_LAB2RGB)


# ── Dataset ───────────────────────────────────────────────────
class RetinaDataset(Dataset):
    def __init__(self, df, img_size, use_clahe=True):
        self.df        = df.reset_index(drop=True)
        self.use_clahe = use_clahe
        self.transform = A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std =[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.use_clahe:
            img = apply_clahe(img)
        img = self.transform(image=img)['image']
        return img, int(row.label)


print('✅ Dataset e transformações prontos')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 5 — Carregar modelo                                  ║
# ╚══════════════════════════════════════════════════════════════╝

def load_model(cfg):
    """
    Carrega o modelo a partir do arquivo resolvido.
    Lida com:
      - state_dict puro (modelo final)
      - dict com chave 'model' (checkpoint completo)
      - prefixo _orig_mod. do torch.compile
    """
    path = cfg['resolved_path']
    arch = cfg['arch']

    # ── Arquitetura ───────────────────────────────────────────
    if cfg['head'] == 'direct':
        model = timm.create_model(arch, pretrained=False, num_classes=5)
    else:
        model = timm.create_model(arch, pretrained=False, num_classes=0)
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(model.num_features, 5)
        )

    # ── Carregar pesos ────────────────────────────────────────
    ckpt = torch.load(path, map_location=device)

    # Aceita checkpoint completo ou state_dict direto
    sd = ckpt.get('model', ckpt) if isinstance(ckpt, dict) and 'model' in ckpt else ckpt

    # Remove prefixo do torch.compile
    sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}

    model.load_state_dict(sd, strict=True)
    model.eval()
    return model.to(device)


print('✅ Função load_model pronta')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 6 — Inferência + métricas                            ║
# ╚══════════════════════════════════════════════════════════════╝

@torch.no_grad()
def run_inference(model, loader):
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in tqdm(loader, desc='  Inferindo', leave=False):
        images = images.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(images)
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(probs.argmax(axis=1))
        all_probs.append(probs)
        del images, logits, probs
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return (
        np.array(all_labels),
        np.array(all_preds),
        np.vstack(all_probs)
    )


def compute_metrics(labels, preds, probs):
    acc          = accuracy_score(labels, preds)
    kappa        = cohen_kappa_score(labels, preds, weights='quadratic')
    prec, rec, f1, sup = precision_recall_fscore_support(
        labels, preds, average=None, labels=list(range(N_CLASSES)), zero_division=0
    )
    f1_macro     = precision_recall_fscore_support(labels, preds, average='macro',    zero_division=0)[2]
    f1_weighted  = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)[2]
    labels_bin   = label_binarize(labels, classes=list(range(N_CLASSES)))
    try:
        auc_macro = roc_auc_score(labels_bin, probs, average='macro', multi_class='ovr')
    except Exception:
        auc_macro = float('nan')
    cm = confusion_matrix(labels, preds, labels=list(range(N_CLASSES)))
    return dict(
        labels=labels, preds=preds, probs=probs,
        acc=acc, kappa=kappa,
        prec=prec, rec=rec, f1=f1, sup=sup,
        f1_macro=f1_macro, f1_weighted=f1_weighted,
        auc_macro=auc_macro, cm=cm
    )


print('✅ Funções de inferência e métricas prontas')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 7 — Rodar validação (detecta modelos disponíveis)    ║
# ╚══════════════════════════════════════════════════════════════╝

results = {}

for model_name, cfg in MODEL_CONFIGS.items():

    if cfg['resolved_path'] is None:
        print(f'⏭️  Pulando {model_name} — arquivo não encontrado')
        continue

    print(f'\n{"="*58}')
    print(f'🔍 Validando {model_name}')
    print(f'   Arquivo : {cfg["resolved_path"]}')
    print(f'   IMG     : {cfg["img_size"]}px | Device: {device}')

    # Dataset e loader para o tamanho correto do modelo
    val_dataset = RetinaDataset(val_df, cfg['img_size'], use_clahe=True)
    val_loader  = DataLoader(
        val_dataset, batch_size=VAL_BATCH, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=False,
        persistent_workers=False
    )
    print(f'   Batches de validação: {len(val_loader)} ({len(val_dataset)} imagens)')

    # Carregar modelo
    try:
        t0 = time.time()
        model = load_model(cfg)
        n_params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f'   ✅ Modelo carregado — {n_params:.1f}M parâmetros ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'   ❌ Erro ao carregar modelo: {e}')
        continue

    # Inferência
    t0 = time.time()
    labels, preds, probs = run_inference(model, val_loader)
    print(f'   ✅ Inferência concluída ({time.time()-t0:.1f}s)')

    # Métricas
    r = compute_metrics(labels, preds, probs)
    r['color'] = cfg['color']
    results[model_name] = r

    print(f'   Acc={r["acc"]:.4f} | Kappa={r["kappa"]:.4f} | '
          f'F1-macro={r["f1_macro"]:.4f} | AUC={r["auc_macro"]:.4f}')

    # Libera VRAM antes do próximo modelo
    del model
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print(f'\n{"="*58}')
print(f'✅ Validação concluída! Modelos avaliados: {list(results.keys())}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 8 — Classification Report completo                   ║
# ╚══════════════════════════════════════════════════════════════╝

for name, r in results.items():
    print(f'\n📋 {name} — Classification Report')
    print('─' * 62)
    print(classification_report(
        r['labels'], r['preds'],
        target_names=CLASS_NAMES, digits=4
    ))
    print(f'  Kappa Quadrático: {r["kappa"]:.4f}')
    print(f'  AUC-ROC (macro) : {r["auc_macro"]:.4f}')
    print('─' * 62)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 9 — Matrizes de Confusão                             ║
# ╚══════════════════════════════════════════════════════════════╝

n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(7.5 * n_models, 6.5))
axes = [axes] if n_models == 1 else axes

for ax, (name, r) in zip(axes, results.items()):
    cm       = r['cm']
    cm_norm  = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    sns.heatmap(
        cm_norm, annot=False, ax=ax,
        cmap='Blues', vmin=0, vmax=1,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, linecolor='white',
        cbar_kws={'label': 'Proporção'}
    )

    # Anotação dupla: proporção + contagem
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            val   = cm_norm[i, j]
            count = cm[i, j]
            color = 'white' if val > 0.5 else 'black'
            ax.text(j + 0.5, i + 0.38, f'{val:.2f}',
                    ha='center', va='center', fontsize=10,
                    fontweight='bold', color=color)
            ax.text(j + 0.5, i + 0.65, f'n={count}',
                    ha='center', va='center', fontsize=7.5,
                    color=color, alpha=0.8)

    ax.set_title(
        f'{name}\n'
        f'Acc={r["acc"]:.4f}  |  Kappa={r["kappa"]:.4f}  |  F1-macro={r["f1_macro"]:.4f}',
        fontsize=11, fontweight='bold', pad=10
    )
    ax.set_ylabel('Classe Real', fontsize=10)
    ax.set_xlabel('Classe Predita', fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Matrizes de Confusão — Retinopatia Diabética',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
path_cm = os.path.join(OUTPUT_DIR, 'confusion_matrices.png')
plt.savefig(path_cm, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Salvo: {path_cm}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 10 — Curvas ROC por classe                           ║
# ╚══════════════════════════════════════════════════════════════╝

palette = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']

fig, axes = plt.subplots(1, n_models, figsize=(8 * n_models, 6))
axes = [axes] if n_models == 1 else axes

for ax, (name, r) in zip(axes, results.items()):
    labels_bin = label_binarize(r['labels'], classes=list(range(N_CLASSES)))
    aucs = []
    for i, cname in enumerate(CLASS_NAMES):
        try:
            fpr, tpr, _ = roc_curve(labels_bin[:, i], r['probs'][:, i])
            auc_val = roc_auc_score(labels_bin[:, i], r['probs'][:, i])
            ax.plot(fpr, tpr, color=palette[i], lw=2.2,
                    label=f'{cname}  (AUC={auc_val:.3f})')
            aucs.append(auc_val)
        except Exception:
            aucs.append(float('nan'))

    ax.fill_between([0,1], [0,1], alpha=0.05, color='gray')
    ax.plot([0,1], [0,1], 'k--', lw=1, alpha=0.4, label='Aleatório')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('Taxa de Falso Positivo (FPR)', fontsize=10)
    ax.set_ylabel('Taxa de Verdadeiro Positivo (TPR)', fontsize=10)
    ax.set_title(
        f'Curvas ROC — {name}\nAUC-macro = {r["auc_macro"]:.4f}',
        fontsize=11, fontweight='bold'
    )
    ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.25)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Curvas ROC por Classe (One-vs-Rest) — Retinopatia Diabética',
             fontsize=13, fontweight='bold')
plt.tight_layout()
path_roc = os.path.join(OUTPUT_DIR, 'roc_curves.png')
plt.savefig(path_roc, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Salvo: {path_roc}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 11 — F1 por classe (comparativo)                     ║
# ╚══════════════════════════════════════════════════════════════╝

x     = np.arange(N_CLASSES)
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

offsets = np.linspace(-(len(results)-1)/2, (len(results)-1)/2, len(results)) * width

for offset, (name, r) in zip(offsets, results.items()):
    bars = ax.bar(x + offset, r['f1'], width,
                  label=name, color=r['color'],
                  alpha=0.85, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, r['f1']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.008,
                f'{val:.3f}',
                ha='center', va='bottom',
                fontsize=8.5, fontweight='bold')

# Linhas de F1-macro
for name, r in results.items():
    ax.axhline(r['f1_macro'], color=r['color'],
               linestyle='--', linewidth=1.4, alpha=0.65,
               label=f'{name} — F1-macro={r["f1_macro"]:.3f}')

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel('F1-Score', fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title('F1-Score por Classe — EfficientNet B0 vs B4',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
path_f1 = os.path.join(OUTPUT_DIR, 'f1_por_classe.png')
plt.savefig(path_f1, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Salvo: {path_f1}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 12 — Radar chart (perfil completo)                   ║
# ╚══════════════════════════════════════════════════════════════╝

if n_models >= 2:
    radar_labels = [
        'Acurácia', 'F1-Macro', 'F1-Weighted', 'AUC-ROC', 'Kappa',
        'F1 No DR', 'F1 Mild', 'F1 Moderate', 'F1 Severe', 'F1 Prolif.'
    ]

    def get_radar(r):
        return [r['acc'], r['f1_macro'], r['f1_weighted'],
                r['auc_macro'], max(0, r['kappa']),
                *r['f1']]

    N      = len(radar_labels)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

    for name, r in results.items():
        vals = get_radar(r) + get_radar(r)[:1]
        ax.plot(angles, vals, 'o-', lw=2, label=name, color=r['color'])
        ax.fill(angles, vals, alpha=0.12, color=r['color'])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'],
                       fontsize=8, color='grey')
    ax.grid(True, alpha=0.35)
    ax.set_title('Radar — Perfil Completo de Métricas',
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.32, 1.1), fontsize=11)

    plt.tight_layout()
    path_radar = os.path.join(OUTPUT_DIR, 'radar_comparison.png')
    plt.savefig(path_radar, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Salvo: {path_radar}')
else:
    print('ℹ️  Radar disponível apenas com 2 modelos validados')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 13 — Taxa de erro por classe                         ║
# ╚══════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 5))
axes = [axes] if n_models == 1 else axes

for ax, (name, r) in zip(axes, results.items()):
    errors       = r['labels'] != r['preds']
    total_cls    = np.array([(r['labels'] == i).sum() for i in range(N_CLASSES)])
    error_cnt    = np.array([(( r['labels'] == i) & errors).sum() for i in range(N_CLASSES)])
    error_rate   = error_cnt / total_cls.clip(min=1)

    bars = ax.barh(CLASS_NAMES, error_rate * 100,
                   color=r['color'], alpha=0.82, edgecolor='white')
    for bar, rate, cnt in zip(bars, error_rate, error_cnt):
        ax.text(bar.get_width() + 0.5,
                bar.get_y() + bar.get_height()/2,
                f'{rate*100:.1f}%  (n={int(cnt)})',
                va='center', fontsize=9.5)

    ax.set_xlabel('Taxa de Erro (%)', fontsize=10)
    ax.set_xlim(0, 110)
    ax.set_title(f'Erros por Classe\n{name}', fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Taxa de Erro por Classe — Onde o Modelo Erra',
             fontsize=13, fontweight='bold')
plt.tight_layout()
path_err = os.path.join(OUTPUT_DIR, 'taxa_erro_por_classe.png')
plt.savefig(path_err, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Salvo: {path_err}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 14 — Distribuição de confiança (acertos vs erros)    ║
# ╚══════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 4.5))
axes = [axes] if n_models == 1 else axes

for ax, (name, r) in zip(axes, results.items()):
    max_prob = r['probs'].max(axis=1)
    correct  = r['labels'] == r['preds']

    ax.hist(max_prob[correct],  bins=30, alpha=0.72,
            color='#2ca02c', label=f'Corretas (n={correct.sum()})', density=True)
    ax.hist(max_prob[~correct], bins=30, alpha=0.72,
            color='#d62728', label=f'Erradas  (n={(~correct).sum()})', density=True)

    m_cor = max_prob[correct].mean()
    m_err = max_prob[~correct].mean() if (~correct).sum() > 0 else 0

    ax.axvline(m_cor, color='#2ca02c', ls='--', lw=1.8,
               label=f'Média correta: {m_cor:.3f}')
    ax.axvline(m_err, color='#d62728', ls='--', lw=1.8,
               label=f'Média errada : {m_err:.3f}')

    ax.set_xlabel('Confiança (probabilidade máxima)', fontsize=10)
    ax.set_ylabel('Densidade', fontsize=10)
    ax.set_title(f'Confiança das Predições\n{name}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8.5, framealpha=0.9)
    ax.grid(True, alpha=0.25)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Distribuição de Confiança: Acertos vs Erros',
             fontsize=13, fontweight='bold')
plt.tight_layout()
path_conf = os.path.join(OUTPUT_DIR, 'confianca_predicoes.png')
plt.savefig(path_conf, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Salvo: {path_conf}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 15 — Amostras de erros por modelo                    ║
# ╚══════════════════════════════════════════════════════════════╝
# Mostra as 10 predições com MAIOR confiança mas ERRADAS
# (os "erros confiantes" — os mais preocupantes clinicamente)

for name, r in results.items():
    wrong_mask   = r['labels'] != r['preds']
    wrong_conf   = r['probs'].max(axis=1)[wrong_mask]
    wrong_idx    = np.where(wrong_mask)[0]

    if len(wrong_idx) == 0:
        print(f'{name}: sem erros! 🎉')
        continue

    top10_local = np.argsort(wrong_conf)[::-1][:10]
    top10_global = wrong_idx[top10_local]

    print(f'\n⚠️  {name} — Top-10 erros mais confiantes:')
    print(f'{"Idx":>6} {"Real":>14} {"Predito":>14} {"Confiança":>10}')
    print('─' * 50)
    for gi in top10_global:
        true_cls = CLASS_NAMES[r['labels'][gi]]
        pred_cls = CLASS_NAMES[r['preds'][gi]]
        conf     = r['probs'][gi].max()
        print(f'{gi:>6} {true_cls:>14} {pred_cls:>14} {conf:>10.4f}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CÉLULA 16 — Resumo final consolidado                        ║
# ╚══════════════════════════════════════════════════════════════╝

SEP = '=' * 62
sep = '─' * 62

print(f'\n{SEP}')
print('🏆  RESUMO FINAL — EfficientNet B0 vs B4')
print(f'{SEP}\n')

# ── Métricas globais ──────────────────────────────────────────
header = f'{"Métrica":<22}'
for n in results:
    header += f'{n.split("-")[1]:>16}'
print(header)
print(sep)

global_metrics = [
    ('Acurácia',       lambda r: r['acc']),
    ('F1-Macro',       lambda r: r['f1_macro']),
    ('F1-Weighted',    lambda r: r['f1_weighted']),
    ('AUC-ROC (OvR)',  lambda r: r['auc_macro']),
    ('Kappa Quadr.',   lambda r: r['kappa']),
]
for label, fn in global_metrics:
    vals = [fn(r) for r in results.values()]
    best = max(vals)
    row  = f'{label:<22}'
    for v in vals:
        star = ' ★' if abs(v - best) < 1e-9 else '  '
        row += f'{v:.4f}{star:>6}'
    print(row)

print(sep)

# ── F1 por classe ─────────────────────────────────────────────
print(f'\n{"Classe":<22}', end='')
for n in results:
    print(f'{n.split("-")[1]:>16}', end='')
print()
print(sep)

for ci, cn in enumerate(CLASS_NAMES):
    vals = [r['f1'][ci] for r in results.values()]
    best = max(vals)
    row  = f'{cn:<22}'
    for v in vals:
        star = ' ★' if abs(v - best) < 1e-9 else '  '
        row += f'{v:.4f}{star:>6}'
    print(row)

print(f'\n{SEP}')
print('★  = melhor valor na linha')

# ── Veredicto ─────────────────────────────────────────────────
if n_models >= 2:
    names = list(results.keys())
    score = {n: 0 for n in names}
    for label, fn in global_metrics:
        vals = {n: fn(r) for n, r in results.items()}
        winner = max(vals, key=vals.get)
        score[winner] += 1
    for ci in range(N_CLASSES):
        vals = {n: r['f1'][ci] for n, r in results.items()}
        winner = max(vals, key=vals.get)
        score[winner] += 1

    print(f'\n📊 Pontuação geral (estrelas ganhas):')
    for n, s in score.items():
        print(f'   {n}: {s}/{len(global_metrics)+N_CLASSES} métricas melhores')
    overall_winner = max(score, key=score.get)
    print(f'\n🥇 Melhor modelo geral: {overall_winner}')

print(f'\n{SEP}')
print('\n📁 Gráficos salvos em:')
print(f'   {OUTPUT_DIR}')
for fname in ['confusion_matrices.png','roc_curves.png','f1_por_classe.png',
              'radar_comparison.png','taxa_erro_por_classe.png','confianca_predicoes.png']:
    full = os.path.join(OUTPUT_DIR, fname)
    mark = '✅' if os.path.exists(full) else '─'
    print(f'   {mark} {fname}')

print(f'\n{SEP}')